# SelectOmics quickstart · GBM microRNA

The shortest useful path: load a real omics matrix, run the pipeline, read the
recommendation. Roughly **5 minutes** on a laptop.

**Dataset.** TCGA glioblastoma, 244 samples, 325 microRNA features, 5 subtypes.
The smallest layer in the cohort, chosen so this finishes quickly. The class
sizes are uneven (61 / 46 / 74 / 20 / 43, a 3.7:1 ratio) and the smallest class
has 20 samples, which is what makes the sample-adequacy warnings in Section 4
worth reading rather than skipping.

**What this notebook does not cover.** Baseline comparisons, step ablation,
multi-omics integration, and the full evaluation surface are in
`../GS-OV/OV_SelectOmics_Showcase.ipynb`. Start there once this makes sense.

## 1 · Setup

In [ ]:
# Adds the package to sys.path for this session and registers it with pip for
# later ones. No kernel restart needed.
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

_pkg_root = str(Path("../..").resolve())
if _pkg_root not in sys.path:
    sys.path.insert(0, _pkg_root)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", _pkg_root],
               capture_output=True, text=True)

import numpy as np
import pandas as pd

import SelectOmics
from SelectOmics import SelectOmicsConfig, SelectOmicsPipeline

SelectOmics.enable_logging("INFO")     # the pipeline reports through logging
print(f"SelectOmics {SelectOmics.__version__} · Python {sys.version.split()[0]}")

## 2 · Data

The `_aligned.csv` files are stored features-by-samples, so they are transposed
on load. Labels arrive in a separate file, aligned by row order.

In [ ]:
DATA_DIR = Path(".").resolve()
TARGET   = "Label"

# The MLOmics data is not committed to this repository. mlomics_data fetches it
# from a pinned, immutable dataset revision on first use, verifies each file
# against a recorded SHA-256, and caches it under examples/.mlomics_cache/.
# See examples/mlomics_data.py for the source, the licence and the citation.
if str(DATA_DIR.parent) not in sys.path:
    sys.path.insert(0, str(DATA_DIR.parent))
from mlomics_data import build_input

CSV = DATA_DIR / "GBM_miRNA_quickstart.csv"

if not CSV.exists():
    build_input("GBM", "miRNA").to_csv(CSV, index=False)
    print(f"built {CSV.name} from the pinned MLOmics revision")

df = pd.read_csv(CSV)
n_samples, n_features = df.shape[0], df.shape[1] - 1
print(f"{n_samples} samples x {n_features} features")
print(f"n/p ratio    : {n_samples / n_features:.2f}")
print(f"missing      : {100 * df.drop(columns=[TARGET]).isna().mean().mean():.3f}%")

counts = df[TARGET].value_counts().sort_index()
print("\nclass balance:")
for cls, k in counts.items():
    print(f"  class {cls}: {k:>3}  ({100 * k / n_samples:4.1f}%)")
print(f"imbalance    : {counts.max() / counts.min():.2f}:1")

## 3 · Configure

`suggest()` inspects the file and proposes settings, reporting what it saw and
why. Take it as a starting point and override what you disagree with, which is
what the explicit arguments below do.

In [ ]:
suggested = SelectOmicsConfig.suggest(str(CSV), TARGET)
print(f"suggested algorithm        : {suggested.algorithm}")
print(f"suggested consensus models : {suggested.n_consensus_models}")
print(f"suggested tuning iterations: {suggested.quick_tune_iterations}")

In [ ]:
config = SelectOmicsConfig(
    data_path=str(CSV),
    target_column=TARGET,
    algorithm="RF",
    output_dir="results_quickstart",

    # Kept small so the notebook finishes quickly. Raise both for real work.
    n_consensus_models=5,
    quick_tune_iterations=10,
    n_bootstrap=30,

    # Step 3 needs at least 30 features from Step 2 to engage. On this layer
    # Step 2 hands over roughly 150, so the whole cascade runs and Step 3 does
    # the bulk of the reduction. On narrower inputs it skips instead, and
    # Section 6 shows how to tell which happened.
    enable_step3=True,

    # The evaluation surface. All three are on by default; named here so it is
    # obvious what produces the output in Sections 4 and 5.
    enable_step_evaluations=True,
    enable_final_test_evaluation=True,
    create_visualizations=True,
    save_intermediate_results=True,
    verbose=True,
)
print(f"output directory: {config.output_dir}")
print(f"algorithm       : {config.algorithm}")
print(f"consensus models: {config.n_consensus_models}")

## 4 · Run

`validate=True` runs the three validation protocols (stratified CV,
leave-one-out, bootstrap) against **every** step's panel, then builds the
recommendation from that comparison. Without it you get panels but no verdict.

In [ ]:
pipeline = SelectOmicsPipeline(config)
results  = pipeline.run(validate=True)
print("\nresult keys:", sorted(results.keys()))

## 5 · What came back

Two accessors, and they answer different questions.

`get_selected_features()` returns the **last** step that ran. That is the right
default when you want the end of the pipeline.

`get_recommended_features()` returns the panel the validation **recommends**:
the smallest one whose score is within one standard error of the best.

On this dataset they agree. They come apart when a later step validates
clearly worse than an earlier one, which Section 6 explains.

In [ ]:
last        = pipeline.get_selected_features()
recommended = pipeline.get_recommended_features()

print(f"last step's panel : {len(last):>3} features")
print(f"recommended panel : {len(recommended):>3} features")
print(f"identical         : {set(last) == set(recommended)}")

rec = results["recommendation"]
print(f"\nrecommended step  : {rec['step_id']}")
print(f"weighted AUC      : {rec['weighted_auc']:.4f}")
print(f"quality           : {rec['quality']}")
print(f"reason            : {rec['reason']}")

In [ ]:
# How each step's panel scored under all three protocols. This table is what
# the recommendation is computed from, so it is worth reading directly.
comparison = results["validation"]["comparison_df"]
cols = ["n_features", "cv_auc", "cv_std", "loo_auc", "bootstrap_auc",
        "bootstrap_ci_lower", "bootstrap_ci_upper"]
display(comparison[[c for c in cols if c in comparison.columns]].round(4))

## 6 · The cascade, and when to distrust its last step

Each step is more aggressive than the last, and on this layer all three run:
roughly 325 features in, about 150 surviving Steps 1 and 2, and Step 3 doing
the bulk of the reduction from there.

Step 3 eliminates most of what it receives. That is correct when its input is
mostly noise. It is wrong when the input is already clean, where it keeps
returning only true features but too few of them, and the panel gets smaller
without getting better.

The recommendation falls back to an earlier panel when a later step validates
clearly worse, and `get_recommended_features()` then returns it. It cannot see a
small loss: every panel is validated on the samples its features were selected
from, so the scores are usually too close to separate, and the smaller panel
wins. Measured with nested CV across seven datasets, following it kept held-out
AUC within about 0.01 of using every feature (`benchmarks/BENCHMARKS.md`,
section 8). Set `enable_nested_cv=True` to check that on your own data; the
showcase notebook shows what it reports.

In [ ]:
per_step = []
for key in ("step0", "step1", "step2", "step3"):
    if key not in results:
        continue
    if results[key].get("skipped"):
        per_step.append({"step": key, "n_features": None, "note": "skipped"})
        continue
    per_step.append({
        "step": key,
        "n_features": len(pipeline.get_selected_features(key)),
        "note": results[key].get("consensus_outcome", ""),
    })
display(pd.DataFrame(per_step))

if results.get("step3", {}).get("skipped", True):
    print("\nStep 3 did not run: Step 2 returned fewer than 30 features, "
          "which is its entry gate.")

## 7 · Held-out test performance

Everything above used the training split only. This is the first and only look
at the held-out set, and it scores the **recommended** panel: the one you are
told to use, whichever step it came from.

In [ ]:
if "final_test" in results:
    ft = results["final_test"]
    print(f"panel scored   : {ft['panel']}, {ft['step_name']} "
          f"({ft['n_features']} features)")
    print(f"test AUC       : {ft.get('test_auc', float('nan')):.4f}")
    print(f"test samples   : {len(pipeline._y_test)}")
    bal = ft["metrics"].get("balanced_accuracy")
    if bal is not None:
        print(f"balanced acc.  : {bal:.4f}")
else:
    print("No final test evaluation. It runs only under run(validate=True) "
          "with enable_final_test_evaluation=True.")

In [ ]:
# Sample-size adequacy. Worth reading on this dataset: the smallest class has
# 20 samples, which constrains how many CV folds are possible.
adequacy = results.get("sample_adequacy")
if adequacy:
    print(f"overall severity: {adequacy.get('overall_severity')}")
    for w in adequacy.get("warnings", []):
        print(f"  - {w}")

## 8 · Save

`recommended_features.csv` is the panel to carry forward.
`selected_features.csv` is the last step's, which on this dataset is the same
list.

In [ ]:
pipeline.save_results()
out = Path(config.output_dir)
for f in sorted(out.glob("*")):
    print(f"  {f.name:<44} {f.stat().st_size / 1024:>8.1f} KB")

In [ ]:
# The provenance table records which step dropped each feature.
provenance = pipeline.get_feature_provenance()
print(f"{len(provenance)} features tracked")
display(provenance.head(10))

## Next

`../GS-OV/OV_SelectOmics_Showcase.ipynb` covers what this skipped: how the
panel compares to LASSO / ElasticNet / RFECV at equal AUC, what each step
contributes on its own, multi-omics integration across four layers, nested CV,
and reproducing a run exactly.